# AIC 2026 — Mã hoá lại video N bị hỏng frame (Colab L4)

Notebook này chạy `r2_reencode_n_damaged_videos.py` trên GPU **L4**. L4 có NVENC; **A100 thì không**, nên đừng chọn A100.

- Nguồn: bản gốc `aic26-media/Videos/Videos_Nxxx/<id>.mp4` (chỉ đọc).
- Đích: `aic26-media/Videos_Web_v2/Videos_Nxxx/<id>.mp4`.
- Mỗi video: giải mã lenient → mã hoá H.264 bằng NVENC, giữ nguyên timestamp → kiểm giải mã strict sạch, thời điểm từng frame ≤ 2 ms, mọi keyframe trong registry có frame tương ứng → upload.
- Máy local chạy danh sách từ đầu; notebook này chạy **từ cuối** (`--reverse`). Video bên kia đã publish sẽ tự bỏ qua. Chạy lại ô cuối là tiếp tục từ chỗ dừng.

**Chuẩn bị:** Runtime → Change runtime type → **L4 GPU**. Colab Secrets (biểu tượng chìa khoá) cần có `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_ACCOUNT_ID`, đều bật *Notebook access*.

103 video trong danh sách.

## 1. GPU, ffmpeg có NVENC, thư viện

In [ ]:
%%bash
set -e
nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
cd /content
if [ ! -x /content/ffmpeg-n8.1/bin/ffmpeg ]; then
  curl -sSL -o ffmpeg.tar.xz "https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-n8.1-latest-linux64-gpl-8.1.tar.xz"
  mkdir -p /content/ffmpeg-n8.1 && tar -xJf ffmpeg.tar.xz -C /content/ffmpeg-n8.1 --strip-components=1 && rm ffmpeg.tar.xz
fi
/content/ffmpeg-n8.1/bin/ffmpeg -hide_banner -version | head -1
pip install -q boto3 pyarrow

In [ ]:
import os, re, subprocess
os.environ["PATH"] = "/content/ffmpeg-n8.1/bin:" + os.environ["PATH"]
# Colab keeps the NVIDIA driver libraries (libnvidia-encode) here.
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")
smoke = subprocess.run(
    ["ffmpeg", "-hide_banner", "-v", "error", "-f", "lavfi", "-i", "testsrc2=size=1920x1080:rate=25",
     "-t", "2", "-c:v", "h264_nvenc", "-f", "null", "-"],
    capture_output=True, text=True,
)
if smoke.returncode != 0:
    print("\n".join(l for l in smoke.stderr.splitlines() if re.search(r"nvenc|driver|cuda|cannot|error", l, re.I)))
    raise SystemExit("NVENC không chạy được — xem các dòng trên (runtime có phải L4 không?)")
print("NVENC OK, CPU:", os.cpu_count())

## 2. Credential R2 (từ Colab Secrets, không in ra)

In [ ]:
import os
from google.colab import userdata
for name in ("R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_ACCOUNT_ID"):
    value = userdata.get(name)
    assert value, f"Thiếu Colab Secret {name}"
    os.environ[name] = value.strip()
print("R2 credentials loaded")

## 3. Code (nguyên văn từ repo ClueScope)

In [ ]:
%%writefile /content/elastic_upload.py
#!/usr/bin/env python3
"""Create Elasticsearch indices and bulk-upload retrieval metadata.

Sources: local staging JSONL (BTC keyframe map / speech / audio / OCR), final
InfoShot++ map CSVs, and, for the object-detection index, the committed
`od-frame-v5` shards on Cloudflare R2 — read and checksum-verified straight from
the bucket, no local staging step.
"""

from __future__ import annotations

import argparse
import csv
import gzip
import hashlib
import hmac
import io
import json
import math
import os
import re
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from collections.abc import Callable, Iterator
from dataclasses import dataclass
from datetime import datetime, timezone
from itertools import islice
from pathlib import Path
from typing import Any, Protocol


DEFAULT_INDEX_PREFIX = "aic26"
DEFAULT_R2_BUCKET = "aic26-media"
DEFAULT_MAX_BULK_BYTES = 8 * 1024 * 1024
DEFAULT_INFOSHOTPP_MAP_ROOT = Path(
    "/home/bao/Projects/ExtractKeyframe/keyframe_L/infoshootpp/map-keyframes"
)
DEFAULT_INFOSHOTPP_MAP_INDEX = "aic26_keyframe_map_infoshotpp_v1"
EXPECTED_INFOSHOTPP_VIDEOS = 873
EXPECTED_INFOSHOTPP_KEYFRAMES = 1_339_055
EXPECTED_INFOSHOTPP_CATEGORIES = {f"L{number:02d}" for number in range(21, 31)}


def read_secret(path: Path) -> str:
    value = path.read_text(encoding="utf-8").strip()
    if not value:
        raise ValueError(f"{path} is empty")
    return value


def read_r2_credentials(path: Path | None) -> dict[str, str]:
    """Resolve R2 S3 credentials from the environment, else from a `Label: value` file.

    Environment wins so a CI/container run never needs the loose secret file
    (`CloudflareR2/cloudflareR2_api.txt`, which is git-ignored).
    """
    creds = {
        "access_key": os.environ.get("R2_ACCESS_KEY_ID") or os.environ.get("AWS_ACCESS_KEY_ID") or "",
        "secret_key": os.environ.get("R2_SECRET_ACCESS_KEY") or os.environ.get("AWS_SECRET_ACCESS_KEY") or "",
        "account_id": os.environ.get("R2_ACCOUNT_ID") or "",
        "endpoint": os.environ.get("R2_ENDPOINT") or "",
    }
    labels = {
        "access key id": "access_key",
        "secret access key": "secret_key",
        "account id": "account_id",
        "endpoint": "endpoint",
    }
    if path is not None and path.exists() and not (creds["access_key"] and creds["secret_key"]):
        for raw in path.read_text(encoding="utf-8").splitlines():
            label, sep, value = raw.partition(":")
            field = labels.get(label.strip().lower())
            if sep and field and not creds[field]:
                creds[field] = value.strip()
    if not creds["endpoint"] and creds["account_id"]:
        creds["endpoint"] = f"https://{creds['account_id']}.r2.cloudflarestorage.com"
    missing = [name for name in ("access_key", "secret_key", "endpoint") if not creds[name]]
    if missing:
        raise RuntimeError(
            f"Thiếu R2 credentials: {', '.join(missing)}. Đặt R2_ACCESS_KEY_ID/R2_SECRET_ACCESS_KEY/"
            f"R2_ACCOUNT_ID hoặc trỏ --r2-credentials-file tới file 'Label: value'."
        )
    return creds


class ObjectStore(Protocol):
    """The two read operations the OD import needs, so tests can pass a fake."""

    def list_keys(self, prefix: str) -> Iterator[str]: ...

    def get_object(self, key: str) -> bytes: ...


S3_NS = "{http://s3.amazonaws.com/doc/2006-03-01/}"
_EMPTY_PAYLOAD_SHA256 = hashlib.sha256(b"").hexdigest()


def _sign(key: bytes, message: str) -> bytes:
    return hmac.new(key, message.encode("utf-8"), hashlib.sha256).digest()


class R2Client:
    """Read-only S3/SigV4 client for Cloudflare R2, stdlib only.

    The OD import needs exactly two S3 operations (ListObjectsV2 + GetObject), so
    they are signed here rather than adding a boto3 dependency — this uploader is
    documented as runnable with a bare Python install.
    """

    def __init__(self, endpoint: str, bucket: str, access_key: str, secret_key: str, *, region: str = "auto"):
        self.endpoint = endpoint.rstrip("/")
        self.bucket = bucket
        self.access_key = access_key
        self.secret_key = secret_key
        self.region = region

    def _canonical_uri(self, key: str) -> str:
        base = "/" + urllib.parse.quote(self.bucket, safe="")
        return f"{base}/{urllib.parse.quote(key, safe='/')}" if key else base

    def _signed_headers(self, canonical_uri: str, query: dict[str, str], now: datetime) -> dict[str, str]:
        host = urllib.parse.urlsplit(self.endpoint).netloc
        amz_date = now.strftime("%Y%m%dT%H%M%SZ")
        date_stamp = now.strftime("%Y%m%d")
        canonical_query = urllib.parse.urlencode(sorted(query.items()), quote_via=urllib.parse.quote)
        canonical_headers = (
            f"host:{host}\nx-amz-content-sha256:{_EMPTY_PAYLOAD_SHA256}\nx-amz-date:{amz_date}\n"
        )
        signed_headers = "host;x-amz-content-sha256;x-amz-date"
        canonical_request = "\n".join([
            "GET", canonical_uri, canonical_query, canonical_headers, signed_headers, _EMPTY_PAYLOAD_SHA256,
        ])
        scope = f"{date_stamp}/{self.region}/s3/aws4_request"
        string_to_sign = "\n".join([
            "AWS4-HMAC-SHA256",
            amz_date,
            scope,
            hashlib.sha256(canonical_request.encode("utf-8")).hexdigest(),
        ])
        signing_key = _sign(
            _sign(_sign(_sign(f"AWS4{self.secret_key}".encode("utf-8"), date_stamp), self.region), "s3"),
            "aws4_request",
        )
        signature = hmac.new(signing_key, string_to_sign.encode("utf-8"), hashlib.sha256).hexdigest()
        return {
            "Authorization": (
                f"AWS4-HMAC-SHA256 Credential={self.access_key}/{scope}, "
                f"SignedHeaders={signed_headers}, Signature={signature}"
            ),
            "x-amz-content-sha256": _EMPTY_PAYLOAD_SHA256,
            "x-amz-date": amz_date,
        }

    def _get(self, canonical_uri: str, query: dict[str, str], *, timeout: int = 180, retries: int = 5) -> bytes:
        url = self.endpoint + canonical_uri
        if query:
            url += "?" + urllib.parse.urlencode(sorted(query.items()), quote_via=urllib.parse.quote)
        for attempt in range(retries + 1):
            # Re-signed per attempt: an SigV4 signature expires, so a retry that
            # reused the first one would fail authentication instead of the
            # transient error it is meant to recover from.
            request = urllib.request.Request(
                url, method="GET", headers=self._signed_headers(canonical_uri, query, datetime.now(timezone.utc))
            )
            try:
                with urllib.request.urlopen(request, timeout=timeout) as response:
                    return response.read()
            except urllib.error.HTTPError as exc:
                if exc.code in {429, 500, 502, 503, 504} and attempt < retries:
                    time.sleep(min(2**attempt, 30))
                    continue
                snippet = exc.read()[:500].decode("utf-8", errors="replace")
                raise RuntimeError(f"R2 GET {canonical_uri} returned HTTP {exc.code}: {snippet}") from exc
            except (TimeoutError, urllib.error.URLError) as exc:
                if attempt < retries:
                    print(f"Retrying R2 GET {canonical_uri} after transient network error: {exc}")
                    time.sleep(min(2**attempt, 30))
                    continue
                raise
        raise RuntimeError(f"R2 GET {canonical_uri} exhausted retries")

    def get_object(self, key: str) -> bytes:
        return self._get(self._canonical_uri(key), {})

    def list_keys(self, prefix: str) -> Iterator[str]:
        token: str | None = None
        while True:
            query = {"list-type": "2", "prefix": prefix, "max-keys": "1000"}
            if token:
                query["continuation-token"] = token
            root = ET.fromstring(self._get(self._canonical_uri(""), query))
            for contents in root.findall(f"{S3_NS}Contents"):
                key = contents.findtext(f"{S3_NS}Key")
                if key:
                    yield key
            if root.findtext(f"{S3_NS}IsTruncated") != "true":
                return
            token = root.findtext(f"{S3_NS}NextContinuationToken")
            if not token:
                return


def auth_headers(api_key: str, *, ndjson: bool = False) -> dict[str, str]:
    token = api_key.strip()
    if not token.lower().startswith("apikey "):
        token = f"ApiKey {token}"
    content_type = "application/x-ndjson" if ndjson else "application/json"
    return {
        "Authorization": token,
        "Content-Type": content_type,
        "Accept": "application/json",
    }


def endpoint_join(endpoint: str, path: str) -> str:
    return endpoint.rstrip("/") + "/" + path.lstrip("/")


def json_bytes(payload: Any) -> bytes:
    return json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")


def build_bulk_payload(index_name: str, records: list[dict[str, Any]], *, id_field: str) -> bytes:
    lines: list[str] = []
    for record in records:
        doc_id = record[id_field]
        lines.append(json.dumps({"index": {"_index": index_name, "_id": doc_id}}, ensure_ascii=False))
        lines.append(json.dumps(record, ensure_ascii=False, separators=(",", ":")))
    return ("\n".join(lines) + "\n").encode("utf-8")


def submit_category(video_id: str) -> str:
    return video_id.split("_", 1)[0]


def normalize_ocr_record(record: dict[str, Any]) -> dict[str, Any]:
    raw_id = record["id"]
    parts = raw_id.split("/")
    if len(parts) != 3:
        raise ValueError(f"Unexpected OCR id format: {raw_id}")
    _, video_id, frame_name = parts
    keyframe_n = int(frame_name)
    category = record.get("category") or submit_category(video_id)
    submit_cat = submit_category(video_id)

    return {
        "ocr_id": raw_id,
        "submit_keyframe_id": f"{submit_cat}/{video_id}/{keyframe_n:03d}",
        "video_id": video_id,
        "category": category,
        "submit_category": submit_cat,
        "image_path": record.get("image_path"),
        "status": record.get("status"),
        "error": record.get("error"),
        "keyframe_n": keyframe_n,
        "keyframe_name": f"{keyframe_n:03d}",
        "text_clean": record.get("text_clean") or "",
        "text_clean_fold": record.get("text_clean_fold") or "",
        "text_nfc": record.get("text_nfc") or "",
        "clock": record.get("clock"),
        "hour": record.get("hour"),
        "boxes": record.get("boxes") or [],
        "ts": record.get("ts"),
    }


def normalize_od_record(record: dict[str, Any]) -> dict[str, Any]:
    """Attach the repo-wide submit identity to an OD frame document.

    OD documents are keyed by `document_id = "<video_id>:<frame_name>"`, but every
    other index here joins on `submit_keyframe_id`. Deriving it at index time keeps
    the OD index usable from the retrieval app without rewriting the immutable R2
    shards.
    """
    document_id = record.get("document_id")
    video_id = record.get("video_id")
    frame_name = str(record.get("frame_name") or "")
    if not document_id or not video_id or not frame_name.isdigit():
        raise ValueError(f"Unexpected OD document identity: {document_id!r} ({video_id!r}/{frame_name!r})")
    keyframe_n = int(frame_name)
    submit_cat = submit_category(video_id)
    return {
        **record,
        "submit_keyframe_id": f"{submit_cat}/{video_id}/{keyframe_n:03d}",
        "submit_category": submit_cat,
        "keyframe_n": keyframe_n,
    }


def identity_record(record: dict[str, Any]) -> dict[str, Any]:
    return record


@dataclass(frozen=True)
class UploadJob:
    logical_name: str
    index_name: str
    id_field: str
    mapping: dict[str, Any]
    source_label: str
    # Called with the number of leading records to skip (resume), so a source can
    # skip cheaply — the OD source drops whole committed shards without fetching.
    open_records: Callable[[int], Iterator[dict[str, Any]]]
    # Applied when the index already exists: adding properties to a `dynamic:
    # strict` mapping is the only way documents enriched here can be accepted by
    # an index that the OD notebook created with the bare v5 schema.
    mapping_patch: dict[str, Any] | None = None


class ElasticClient:
    def __init__(self, endpoint: str, api_key: str):
        self.endpoint = endpoint.rstrip("/")
        self.api_key = api_key

    def request(
        self,
        method: str,
        path: str,
        *,
        body: bytes | None = None,
        ndjson: bool = False,
        expected: tuple[int, ...] = (200,),
        timeout: int = 120,
        retries: int = 5,
    ) -> tuple[int, bytes]:
        retryable_http = {429, 502, 503, 504}
        for attempt in range(retries + 1):
            request = urllib.request.Request(
                endpoint_join(self.endpoint, path),
                data=body,
                method=method,
                headers=auth_headers(self.api_key, ndjson=ndjson),
            )
            try:
                with urllib.request.urlopen(request, timeout=timeout) as response:
                    data = response.read()
                    status = response.status
            except urllib.error.HTTPError as exc:
                data = exc.read()
                status = exc.code
                if status in retryable_http and attempt < retries:
                    time.sleep(min(2**attempt, 30))
                    continue
            except (TimeoutError, urllib.error.URLError) as exc:
                if attempt < retries:
                    print(f"Retrying {method} {path} after transient network error: {exc}")
                    time.sleep(min(2**attempt, 30))
                    continue
                raise
            if status not in expected:
                snippet = data[:1000].decode("utf-8", errors="replace")
                raise RuntimeError(f"{method} {path} returned HTTP {status}: {snippet}")
            return status, data
        raise RuntimeError(f"{method} {path} exhausted retries")

    def json_request(
        self,
        method: str,
        path: str,
        *,
        payload: Any | None = None,
        expected: tuple[int, ...] = (200,),
        timeout: int = 120,
    ) -> Any:
        body = None if payload is None else json_bytes(payload)
        _, data = self.request(method, path, body=body, expected=expected, timeout=timeout)
        if not data:
            return None
        return json.loads(data.decode("utf-8"))

    def index_exists(self, index_name: str) -> bool:
        status, _ = self.request("HEAD", index_name, expected=(200, 404), timeout=30)
        return status == 200

    def create_index_if_missing(self, index_name: str, mapping: dict[str, Any]) -> str:
        if self.index_exists(index_name):
            return "exists"
        self.json_request("PUT", index_name, payload=mapping, expected=(200,), timeout=120)
        return "created"

    def bulk_ndjson(self, payload: bytes) -> dict[str, Any]:
        _, data = self.request(
            "POST",
            "_bulk?refresh=false",
            body=payload,
            ndjson=True,
            expected=(200,),
            timeout=240,
        )
        result = json.loads(data.decode("utf-8"))
        if result.get("errors"):
            first_error = next(
                (
                    item
                    for item in result.get("items", [])
                    if item.get("index", {}).get("error")
                ),
                None,
            )
            raise RuntimeError(f"Bulk request had item errors: {first_error}")
        return result

    def refresh(self, index_name: str) -> None:
        self.json_request("POST", f"{index_name}/_refresh", expected=(200,), timeout=120)

    def count(self, index_name: str) -> int:
        result = self.json_request("GET", f"{index_name}/_count", expected=(200,), timeout=120)
        return int(result["count"])


def keyword(ignore_above: int = 512) -> dict[str, Any]:
    return {"type": "keyword", "ignore_above": ignore_above}


def text_with_keyword() -> dict[str, Any]:
    return {"type": "text", "fields": {"keyword": keyword(1024)}}


def base_index_settings() -> dict[str, Any]:
    return {}


def keyframe_mapping() -> dict[str, Any]:
    return {
        **base_index_settings(),
        "mappings": {
            "dynamic": False,
            "properties": {
                "keyframe_id": keyword(),
                "frame_id": keyword(),
                "submit_keyframe_id": keyword(),
                "video_id": keyword(),
                "category_hint": keyword(),
                "submit_category": keyword(),
                "keyframe_n": {"type": "integer"},
                "keyframe_name": keyword(),
                "pts_time": {"type": "float"},
                "fps": {"type": "float"},
                "frame_idx": {"type": "integer"},
            },
        },
    }


KEYFRAME_FIELD_PROPS = {
    "keyframe_id": keyword(),
    "submit_keyframe_id": keyword(),
    "keyframe_n": {"type": "integer"},
    "keyframe_pts_time": {"type": "float"},
    "keyframe_frame_idx": {"type": "integer"},
    "start_keyframe_id": keyword(),
    "start_submit_keyframe_id": keyword(),
    "start_keyframe_n": {"type": "integer"},
    "start_keyframe_pts_time": {"type": "float"},
    "start_keyframe_frame_idx": {"type": "integer"},
    "center_keyframe_id": keyword(),
    "center_submit_keyframe_id": keyword(),
    "center_keyframe_n": {"type": "integer"},
    "center_keyframe_pts_time": {"type": "float"},
    "center_keyframe_frame_idx": {"type": "integer"},
    "end_keyframe_id": keyword(),
    "end_submit_keyframe_id": keyword(),
    "end_keyframe_n": {"type": "integer"},
    "end_keyframe_pts_time": {"type": "float"},
    "end_keyframe_frame_idx": {"type": "integer"},
}


def audio_mapping() -> dict[str, Any]:
    return {
        **base_index_settings(),
        "mappings": {
            "dynamic": False,
            "properties": {
                "window_id": keyword(),
                "video_id": keyword(),
                "start": {"type": "float"},
                "end": {"type": "float"},
                "duration": {"type": "float"},
                "center_time": {"type": "float"},
                "glap_idx": {"type": "integer"},
                "tags": {
                    "type": "nested",
                    "properties": {
                        "label": keyword(),
                        "score": {"type": "float"},
                    },
                },
                "tag_labels": keyword(),
                "tag_scores": {"type": "float"},
                "top1_label": keyword(),
                "top1_score": {"type": "float"},
                "caption": text_with_keyword(),
                "has_caption": {"type": "boolean"},
                "caption_quality": keyword(),
                "audio_stoplist_hit": {"type": "boolean"},
                "top1_is_stoplisted": {"type": "boolean"},
                "source_audio_file": keyword(1024),
                **KEYFRAME_FIELD_PROPS,
            },
        },
    }


def speech_mapping() -> dict[str, Any]:
    return {
        **base_index_settings(),
        "mappings": {
            "dynamic": False,
            "properties": {
                "segment_id": keyword(),
                "video_id": keyword(),
                "start": {"type": "float"},
                "end": {"type": "float"},
                "duration": {"type": "float"},
                "center_time": {"type": "float"},
                "text": text_with_keyword(),
                "avg_word_score": {"type": "float"},
                "confidence_bucket": keyword(),
                "segment_role": keyword(),
                "word_count": {"type": "integer"},
                "avg_logprob": {"type": "float"},
                "no_speech_prob": {"type": "float"},
                "source_segment_idx": {"type": "integer"},
                "source_speech_file": keyword(1024),
                **KEYFRAME_FIELD_PROPS,
            },
        },
    }


def ocr_mapping() -> dict[str, Any]:
    return {
        **base_index_settings(),
        "mappings": {
            "dynamic": False,
            "properties": {
                "ocr_id": keyword(1024),
                "submit_keyframe_id": keyword(),
                "video_id": keyword(),
                "category": keyword(),
                "submit_category": keyword(),
                "image_path": keyword(2048),
                "status": keyword(),
                "error": keyword(1024),
                "keyframe_n": {"type": "integer"},
                "keyframe_name": keyword(),
                "text_clean": text_with_keyword(),
                "text_clean_fold": text_with_keyword(),
                "text_nfc": text_with_keyword(),
                "clock": keyword(),
                "hour": {"type": "integer"},
                "boxes": {
                    "type": "nested",
                    "properties": {
                        "text": text_with_keyword(),
                        "box": {"type": "integer"},
                    },
                },
                "ts": {"type": "double"},
            },
        },
    }


# Identity fields this uploader adds on top of the frozen `od-frame-v5` schema.
OD_SUBMIT_IDENTITY_PROPS = {
    "submit_keyframe_id": keyword(),
    "submit_category": keyword(),
    "keyframe_n": {"type": "integer"},
}


def od_frames_mapping() -> dict[str, Any]:
    """Strict `od-frame-v5` mapping — kept field-for-field in sync with
    `od_elastic.index_mapping()` in OD/OD_Kaggle_Notebook.ipynb, plus the submit
    identity fields above. `dynamic: strict` is deliberate: an unexpected field
    must fail the import instead of silently creating an unqueryable mapping."""
    detections_props = {
        "instance_id": {"type": "keyword"},
        "label": {"type": "keyword"},
        "canonical_label": {"type": "keyword"},
        "aliases": {"type": "keyword"},
        "label_group": {"type": "keyword"},
        "conf": {"type": "half_float"},
        "countable": {"type": "boolean"},
        "counted": {"type": "boolean"},
        "count_threshold_used": {"type": "half_float"},
        "count_source": {"type": "keyword"},
        "sources": {"type": "keyword"},
        "source_scores": {"type": "object", "enabled": False},
        "dominant_color": {"type": "keyword"},
        "color_names": {"type": "keyword"},
        "colors": {"type": "object", "enabled": False},
        "color_method": {"type": "keyword"},
        "color_confidence": {"type": "half_float"},
        "color_purity": {"type": "half_float"},
        "color_entropy": {"type": "half_float"},
        "color_delta_e_p50": {"type": "half_float"},
        "color_coverage": {"type": "half_float"},
        "color_reliable": {"type": "boolean"},
        "bbox_norm": {"properties": {axis: {"type": "float"} for axis in ("x1", "y1", "x2", "y2")}},
        "bbox_px": {"properties": {axis: {"type": "integer"} for axis in ("x1", "y1", "x2", "y2")}},
        "center_norm": {"properties": {"x": {"type": "float"}, "y": {"type": "float"}}},
        "size_norm": {
            "properties": {
                "width": {"type": "float"},
                "height": {"type": "float"},
                "area": {"type": "float"},
            }
        },
        "grid7": {"properties": {"row": {"type": "byte"}, "col": {"type": "byte"}, "id": {"type": "keyword"}}},
        "horizontal_zone": {"type": "keyword"},
        "vertical_zone": {"type": "keyword"},
        "position": {"type": "keyword"},
        "position_tags": {"type": "keyword"},
    }
    object_counts_props = {
        "label": {"type": "keyword"},
        "count": {"type": "integer"},
        "countable": {"type": "boolean"},
        "max_conf": {"type": "half_float"},
        "mean_conf": {"type": "half_float"},
        "aliases": {"type": "keyword"},
        "is_estimate": {"type": "boolean"},
        "is_lower_bound": {"type": "boolean"},
        "lower_bound_reasons": {"type": "keyword"},
        "count_quality": {"type": "keyword"},
        "thresholds_used": {"type": "half_float"},
        "sources": {"type": "keyword"},
    }
    return {
        # No `settings` block on purpose: this repo targets Elastic serverless,
        # which rejects number_of_shards/number_of_replicas/refresh_interval and
        # sizes indices itself. `od_elastic.index_mapping()` still sets them for a
        # self-managed cluster; the field mappings below are what must stay in sync.
        **base_index_settings(),
        "mappings": {
            "dynamic": "strict",
            "properties": {
                "schema_version": {"type": "keyword"},
                "pipeline_version": {"type": "keyword"},
                "config_hash": {"type": "keyword"},
                "manifest_hash": {"type": "keyword"},
                "runtime_hash": {"type": "keyword"},
                "resolved_detector": {"type": "keyword"},
                "weight_sha256": {"type": "keyword"},
                "text_encoder_sha256": {"type": "keyword"},
                "prompt_embeddings_sha256": {"type": "keyword"},
                "document_id": {"type": "keyword"},
                "keyframe_id": {"type": "keyword"},
                "video_id": {"type": "keyword"},
                "frame_name": {"type": "keyword"},
                "frame_index": {"type": "integer"},
                "batch": {"type": "keyword"},
                "genre": {"type": "keyword"},
                "r2_key": {"type": "keyword", "index": False, "doc_values": False},
                "image_url": {"type": "keyword", "index": False, "doc_values": False},
                "input_etag": {"type": "keyword"},
                "status": {"type": "keyword"},
                "fallback_status": {"type": "keyword"},
                "fallback_reasons": {"type": "keyword"},
                "fallback_error": {"type": "text", "index": False},
                "error": {"type": "text", "index": False},
                "generated_at": {"type": "date"},
                "width": {"type": "integer"},
                "height": {"type": "integer"},
                "detection_count": {"type": "integer"},
                "counted_detection_count": {"type": "integer"},
                "labels": {"type": "keyword"},
                "canonical_labels": {"type": "keyword"},
                "spatial_tokens": {"type": "keyword"},
                "count_tokens": {"type": "keyword"},
                "object_text": {"type": "text", "analyzer": "standard"},
                "crowd_present": {"type": "boolean"},
                "counts_are_estimates": {"type": "boolean"},
                "count_is_lower_bound": {"type": "boolean"},
                "count_uncertain_labels": {"type": "keyword"},
                "count_notes": {"type": "keyword", "index": False},
                "pipeline": {"type": "object", "enabled": False},
                "object_counts": {"type": "nested", "dynamic": "strict", "properties": object_counts_props},
                "detections": {"type": "nested", "dynamic": "strict", "properties": detections_props},
                **OD_SUBMIT_IDENTITY_PROPS,
            },
        },
    }


OD_MARKER_RE = re.compile(r"part-(\d{6})\.complete\.json$")


def od_committed_shard_ids(r2: ObjectStore, output_prefix: str) -> list[int]:
    """Shard ids that reached the commit point (a completion marker exists).

    A raw `shards/part-*.jsonl.gz` without its marker is an interrupted shard and
    is deliberately ignored — the OD pipeline treats the marker, not the data
    file, as the commit.
    """
    ids = []
    for key in r2.list_keys(f"{output_prefix.rstrip('/')}/complete/"):
        match = OD_MARKER_RE.search(key)
        if match:
            ids.append(int(match.group(1)))
    return sorted(ids)


def iter_od_documents(
    r2: ObjectStore,
    output_prefix: str,
    shard_ids: list[int],
    *,
    skip: int = 0,
    expected_hashes: dict[str, str] | None = None,
    verify_checksum: bool = True,
) -> Iterator[dict[str, Any]]:
    """Stream OD frame documents from committed R2 shards, newest identity checks first.

    Each shard is verified against its completion marker (namespace hashes, gzip
    SHA-256, document count) before any of its documents are indexed, so a
    truncated upload or a mixed-namespace prefix fails loudly instead of writing
    a half-valid index.
    """
    prefix = output_prefix.rstrip("/")
    remaining_skip = max(0, skip)
    for position, shard_id in enumerate(shard_ids, 1):
        marker_key = f"{prefix}/complete/part-{shard_id:06d}.complete.json"
        marker = json.loads(r2.get_object(marker_key).decode("utf-8"))
        document_count = int(marker.get("document_count") or 0)
        mismatched = [
            field
            for field, value in (expected_hashes or {}).items()
            if value and marker.get(field) != value
        ]
        if mismatched:
            raise RuntimeError(f"{marker_key}: namespace mismatch on {', '.join(mismatched)}")
        if int(marker.get("error_count") or 0):
            raise RuntimeError(f"{marker_key}: error_count != 0; shard chưa phải commit hợp lệ")
        if remaining_skip >= document_count:
            remaining_skip -= document_count
            continue
        payload = r2.get_object(f"{prefix}/shards/part-{shard_id:06d}.jsonl.gz")
        if verify_checksum and marker.get("sha256"):
            digest = hashlib.sha256(payload).hexdigest()
            if digest != marker["sha256"]:
                raise RuntimeError(
                    f"Shard part-{shard_id:06d} checksum mismatch: {digest} != {marker['sha256']}"
                )
        emitted = 0
        with gzip.GzipFile(fileobj=io.BytesIO(payload), mode="rb") as stream:
            for raw in stream:
                if not raw.strip():
                    continue
                emitted += 1
                if remaining_skip:
                    remaining_skip -= 1
                    continue
                yield json.loads(raw)
        if emitted != document_count:
            raise RuntimeError(
                f"Shard part-{shard_id:06d}: {emitted} documents != marker document_count {document_count}"
            )
        print(f"[od_frames] shard {position}/{len(shard_ids)} part-{shard_id:06d} ({document_count:,} docs)")


def od_source(
    r2: ObjectStore,
    output_prefix: str,
    *,
    expected_hashes: dict[str, str] | None = None,
    max_shards: int | None = None,
    verify_checksum: bool = True,
) -> Callable[[int], Iterator[dict[str, Any]]]:
    def open_records(skip: int) -> Iterator[dict[str, Any]]:
        shard_ids = od_committed_shard_ids(r2, output_prefix)
        if not shard_ids:
            raise RuntimeError(f"Không có completion marker nào dưới {output_prefix.rstrip('/')}/complete/")
        if max_shards is not None:
            shard_ids = shard_ids[:max_shards]
        print(
            f"[od_frames] committed shards={len(shard_ids)} "
            f"(part-{shard_ids[0]:06d}..part-{shard_ids[-1]:06d})"
        )
        for document in iter_od_documents(
            r2,
            output_prefix,
            shard_ids,
            skip=skip,
            expected_hashes=expected_hashes,
            verify_checksum=verify_checksum,
        ):
            yield normalize_od_record(document)

    return open_records


def jsonl_source(
    path: Path, transform: Callable[[dict[str, Any]], dict[str, Any]] = identity_record
) -> Callable[[int], Iterator[dict[str, Any]]]:
    def open_records(skip: int) -> Iterator[dict[str, Any]]:
        return drop_first(iter_jsonl(path, transform), skip)

    return open_records


def _infoshotpp_map_files(root: Path) -> list[Path]:
    files = sorted(root.glob("L*/L*_V*.csv"))
    if not files:
        raise FileNotFoundError(f"Không tìm thấy map-keyframe CSV dưới {root}")
    return files


def iter_infoshotpp_keyframes(root: Path, *, skip: int = 0) -> Iterator[dict[str, Any]]:
    """Read final InfoShot++ maps without conflating ordinal ``n`` and frame_idx.

    The final JPEG is ``f{frame_idx:08d}.jpg`` while the application identity is
    ``<category>/<video_id>/<n:03d>``.  ``pts_time`` is deliberately not required
    to be monotonic: the final L25 maps contain a handful of sub-frame regressions,
    while frame_idx remains the canonical strictly-increasing decode identity.
    """
    remaining = max(0, skip)
    for path in _infoshotpp_map_files(root):
        video_id = path.stem
        category = path.parent.name
        if video_id.split("_", 1)[0] != category:
            raise ValueError(f"{path}: category/video mismatch")
        expected_n = 1
        previous_frame_idx = -1
        with path.open("r", encoding="utf-8", newline="") as handle:
            reader = csv.DictReader(handle)
            if reader.fieldnames != ["n", "pts_time", "fps", "frame_idx"]:
                raise ValueError(f"{path}: unexpected header {reader.fieldnames!r}")
            for line_number, row in enumerate(reader, 2):
                try:
                    n = int(row["n"])
                    pts_time = float(row["pts_time"])
                    fps = float(row["fps"])
                    frame_idx = int(row["frame_idx"])
                except (KeyError, TypeError, ValueError) as exc:
                    raise ValueError(f"{path}:{line_number}: invalid map row") from exc
                if n != expected_n:
                    raise ValueError(f"{path}:{line_number}: n={n}, expected {expected_n}")
                if frame_idx <= previous_frame_idx:
                    raise ValueError(
                        f"{path}:{line_number}: frame_idx={frame_idx} is not strictly increasing"
                    )
                if not math.isfinite(pts_time) or not math.isfinite(fps) or fps <= 0:
                    raise ValueError(f"{path}:{line_number}: invalid pts_time/fps")
                expected_n += 1
                previous_frame_idx = frame_idx
                if remaining:
                    remaining -= 1
                    continue
                keyframe_name = f"{n:03d}"
                yield {
                    "keyframe_id": f"{video_id}/{keyframe_name}",
                    "frame_id": f"{video_id}@f{frame_idx:08d}",
                    "submit_keyframe_id": f"{category}/{video_id}/{keyframe_name}",
                    "video_id": video_id,
                    "category_hint": category,
                    "submit_category": category,
                    "keyframe_n": n,
                    "keyframe_name": keyframe_name,
                    "pts_time": pts_time,
                    "fps": fps,
                    "frame_idx": frame_idx,
                }


def infoshotpp_source(root: Path) -> Callable[[int], Iterator[dict[str, Any]]]:
    def open_records(skip: int) -> Iterator[dict[str, Any]]:
        return iter_infoshotpp_keyframes(root, skip=skip)

    return open_records


def audit_infoshotpp_maps(root: Path) -> dict[str, Any]:
    """Fully validate the fixed final corpus before the first Elastic write."""
    videos: set[str] = set()
    categories: set[str] = set()
    rows = 0
    for record in iter_infoshotpp_keyframes(root):
        rows += 1
        videos.add(record["video_id"])
        categories.add(record["submit_category"])
    if rows != EXPECTED_INFOSHOTPP_KEYFRAMES:
        raise RuntimeError(
            f"InfoShot++ map rows={rows:,}, expected {EXPECTED_INFOSHOTPP_KEYFRAMES:,}"
        )
    if len(videos) != EXPECTED_INFOSHOTPP_VIDEOS:
        raise RuntimeError(
            f"InfoShot++ videos={len(videos):,}, expected {EXPECTED_INFOSHOTPP_VIDEOS:,}"
        )
    if categories != EXPECTED_INFOSHOTPP_CATEGORIES:
        raise RuntimeError(
            f"InfoShot++ categories={sorted(categories)}, expected "
            f"{sorted(EXPECTED_INFOSHOTPP_CATEGORIES)}"
        )
    result = {"rows": rows, "videos": len(videos), "categories": sorted(categories)}
    print(f"[keyframe_map_infoshotpp] preflight PASS: {result}")
    return result


def iter_jsonl(path: Path, transform: Callable[[dict[str, Any]], dict[str, Any]]) -> Iterator[dict[str, Any]]:
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                yield transform(json.loads(line))


def drop_first(iterator: Iterator[Any], count: int) -> Iterator[Any]:
    return islice(iterator, count, None)


def chunked(
    iterator: Iterator[dict[str, Any]], batch_size: int, max_bytes: int = 0
) -> Iterator[list[dict[str, Any]]]:
    """Batch by record count and, when `max_bytes` is set, by serialized size.

    OCR/speech/audio records are small and uniform, but one OD frame document
    carries every detection with geometry and colour — a flat 2,000-record batch
    can exceed Elasticsearch's HTTP payload limit, so size closes the batch first.
    """
    batch: list[dict[str, Any]] = []
    batch_bytes = 0
    for item in iterator:
        item_bytes = len(json_bytes(item)) + 128 if max_bytes else 0  # +128 ≈ the action line
        if batch and max_bytes and batch_bytes + item_bytes > max_bytes:
            yield batch
            batch, batch_bytes = [], 0
        batch.append(item)
        batch_bytes += item_bytes
        if len(batch) >= batch_size:
            yield batch
            batch, batch_bytes = [], 0
    if batch:
        yield batch


def index_name(prefix: str, suffix: str) -> str:
    return f"{prefix}_{suffix}_v1"


def build_jobs(
    prefix: str,
    staging_dir: Path,
    ocr_path: Path | None,
    infoshotpp_map_root: Path | None = None,
    infoshotpp_map_index: str = DEFAULT_INFOSHOTPP_MAP_INDEX,
    od_job: UploadJob | None = None,
) -> list[UploadJob]:
    jobs = [
        UploadJob(
            logical_name="keyframe_map",
            index_name=index_name(prefix, "keyframe_map"),
            id_field="submit_keyframe_id",
            mapping=keyframe_mapping(),
            source_label=str(staging_dir / "keyframe_map.jsonl"),
            open_records=jsonl_source(staging_dir / "keyframe_map.jsonl"),
        ),
        UploadJob(
            logical_name="speech_segments",
            index_name=index_name(prefix, "speech_segments"),
            id_field="segment_id",
            mapping=speech_mapping(),
            source_label=str(staging_dir / "speech_segments_mapped.jsonl"),
            open_records=jsonl_source(staging_dir / "speech_segments_mapped.jsonl"),
        ),
        UploadJob(
            logical_name="audio_windows",
            index_name=index_name(prefix, "audio_windows"),
            id_field="window_id",
            mapping=audio_mapping(),
            source_label=str(staging_dir / "audio_windows_mapped.jsonl"),
            open_records=jsonl_source(staging_dir / "audio_windows_mapped.jsonl"),
        ),
    ]
    if infoshotpp_map_root is not None:
        jobs.append(
            UploadJob(
                logical_name="keyframe_map_infoshotpp",
                index_name=infoshotpp_map_index,
                id_field="submit_keyframe_id",
                mapping=keyframe_mapping(),
                source_label=str(infoshotpp_map_root),
                open_records=infoshotpp_source(infoshotpp_map_root),
            )
        )
    if ocr_path is not None:
        jobs.append(
            UploadJob(
                logical_name="ocr_keyframes",
                index_name=index_name(prefix, "ocr_keyframes"),
                id_field="submit_keyframe_id",
                mapping=ocr_mapping(),
                source_label=str(ocr_path),
                open_records=jsonl_source(ocr_path, normalize_ocr_record),
            )
        )
    if od_job is not None:
        jobs.append(od_job)
    return jobs


def build_od_job(args: argparse.Namespace) -> UploadJob | None:
    """OD frames job reading committed shards straight from R2 (no local staging)."""
    output_prefix = args.od_prefix
    expected_hashes: dict[str, str] = {}
    if args.od_run_summary is not None and args.od_run_summary.exists():
        summary = json.loads(args.od_run_summary.read_text(encoding="utf-8"))
        output_prefix = output_prefix or summary.get("output_prefix")
        expected_hashes = {
            field: summary[field]
            for field in ("config_hash", "manifest_hash", "runtime_hash")
            if summary.get(field)
        }
    if not output_prefix:
        return None
    creds = read_r2_credentials(args.r2_credentials_file)
    r2 = R2Client(creds["endpoint"], args.r2_bucket, creds["access_key"], creds["secret_key"])
    return UploadJob(
        logical_name="od_frames",
        index_name=args.od_index or index_name(args.index_prefix, "od_frames"),
        id_field="document_id",
        mapping=od_frames_mapping(),
        source_label=f"r2://{args.r2_bucket}/{output_prefix.rstrip('/')}",
        open_records=od_source(
            r2,
            output_prefix,
            expected_hashes=expected_hashes,
            max_shards=args.od_max_shards,
            verify_checksum=not args.od_skip_checksum,
        ),
        mapping_patch={"properties": OD_SUBMIT_IDENTITY_PROPS},
    )


def upload_job(
    client: ElasticClient,
    job: UploadJob,
    batch_size: int,
    *,
    resume_existing: bool = False,
    max_bulk_bytes: int = DEFAULT_MAX_BULK_BYTES,
) -> dict[str, Any]:
    status = client.create_index_if_missing(job.index_name, job.mapping)
    if status == "exists" and job.mapping_patch:
        client.json_request("PUT", f"{job.index_name}/_mapping", payload=job.mapping_patch, timeout=60)
    skipped = 0
    if resume_existing and status == "exists":
        client.refresh(job.index_name)
        skipped = client.count(job.index_name)
    uploaded = 0
    took_ms = 0
    started = time.time()
    print(f"[{job.index_name}] index {status}; source={job.source_label}; skip={skipped:,}")
    records = job.open_records(skipped)
    for batch in chunked(records, batch_size, max_bulk_bytes):
        payload = build_bulk_payload(job.index_name, batch, id_field=job.id_field)
        result = client.bulk_ndjson(payload)
        uploaded += len(batch)
        took_ms += int(result.get("took", 0))
        source_position = skipped + uploaded
        if source_position % (batch_size * 10) == 0:
            print(f"[{job.index_name}] uploaded source position {source_position:,}")
    client.refresh(job.index_name)
    remote_count = client.count(job.index_name)
    elapsed = round(time.time() - started, 2)
    print(
        f"[{job.index_name}] uploaded_new={uploaded:,}; skipped={skipped:,}; "
        f"remote_count={remote_count:,}; elapsed={elapsed}s"
    )
    return {
        "index": job.index_name,
        "logical_name": job.logical_name,
        "index_status": status,
        "source": job.source_label,
        "uploaded": uploaded,
        "skipped_existing": skipped,
        "remote_count": remote_count,
        "bulk_took_ms": took_ms,
        "elapsed_s": elapsed,
    }


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--endpoint-file", type=Path, default=Path("API_KEY/elastic_endpoint.txt"))
    parser.add_argument("--api-key-file", type=Path, default=Path("API_KEY/elastic_apikey.txt"))
    parser.add_argument("--staging-dir", type=Path, default=Path("elastic_staging"))
    parser.add_argument("--ocr-path", type=Path, default=Path("OCR/ocr_clean.jsonl"))
    parser.add_argument("--no-ocr", action="store_true")
    parser.add_argument(
        "--infoshootpp-map-root",
        type=Path,
        default=DEFAULT_INFOSHOTPP_MAP_ROOT,
        help="Root chứa Lxx/Lxx_Vxxx.csv của InfoShot++ final map.",
    )
    parser.add_argument(
        "--infoshootpp-map-index",
        default=DEFAULT_INFOSHOTPP_MAP_INDEX,
        help="Index riêng cho map InfoShot++; không ghi đè map BTC.",
    )
    parser.add_argument("--no-infoshootpp-map", action="store_true")
    parser.add_argument("--index-prefix", default=DEFAULT_INDEX_PREFIX)
    parser.add_argument(
        "--only",
        nargs="*",
        choices=[
            "keyframe_map",
            "keyframe_map_infoshotpp",
            "speech_segments",
            "audio_windows",
            "ocr_keyframes",
            "od_frames",
        ],
        help="Upload only selected logical indices.",
    )
    parser.add_argument("--resume-existing", action="store_true")
    parser.add_argument("--batch-size", type=int, default=2000)
    parser.add_argument("--max-bulk-bytes", type=int, default=DEFAULT_MAX_BULK_BYTES)
    parser.add_argument("--summary-path", type=Path, default=Path("elastic_staging/elastic_upload_summary.json"))

    od = parser.add_argument_group("object detection (đọc shard đã commit trên Cloudflare R2)")
    od.add_argument(
        "--od-prefix",
        help="R2 output prefix, vd Derived/ObjectDetection/aic26-od-v5/<config>/input-<manifest>/runtime-<runtime>",
    )
    od.add_argument(
        "--od-run-summary",
        type=Path,
        help="JSON summary của run OD (chứa output_prefix + *_hash); dùng thay/bổ sung cho --od-prefix",
    )
    od.add_argument("--od-index", help=f"Mặc định {index_name(DEFAULT_INDEX_PREFIX, 'od_frames')}")
    od.add_argument("--od-max-shards", type=int, help="Chỉ import N shard đầu (chạy thử)")
    od.add_argument("--od-skip-checksum", action="store_true", help="Bỏ verify SHA-256 của shard")
    od.add_argument("--r2-bucket", default=DEFAULT_R2_BUCKET)
    od.add_argument(
        "--r2-credentials-file",
        type=Path,
        default=Path("CloudflareR2/cloudflareR2_api.txt"),
        help="File 'Label: value' (Access Key ID / Secret Access Key / Account ID); env R2_* được ưu tiên",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    endpoint = read_secret(args.endpoint_file)
    api_key = read_secret(args.api_key_file)
    ocr_path = None if args.no_ocr else args.ocr_path
    infoshotpp_root = None if args.no_infoshootpp_map else args.infoshootpp_map_root
    jobs = build_jobs(
        args.index_prefix,
        args.staging_dir,
        ocr_path,
        infoshotpp_root,
        args.infoshootpp_map_index,
        build_od_job(args),
    )
    if args.only:
        allowed = set(args.only)
        jobs = [job for job in jobs if job.logical_name in allowed]
    if any(job.logical_name == "keyframe_map_infoshotpp" for job in jobs):
        # Intentional second read: this pass proves the whole corpus is valid;
        # upload_job then streams it. No partially-audited index is created.
        audit_infoshotpp_maps(args.infoshootpp_map_root)

    client = ElasticClient(endpoint, api_key)
    cluster_info = client.json_request("GET", "/", expected=(200,), timeout=60)
    print(
        "Connected to Elasticsearch",
        json.dumps(
            {
                "cluster_name": cluster_info.get("cluster_name"),
                "version": (cluster_info.get("version") or {}).get("number"),
                "tagline": cluster_info.get("tagline"),
            },
            ensure_ascii=False,
        ),
    )

    results = [
        upload_job(
            client,
            job,
            args.batch_size,
            resume_existing=args.resume_existing,
            max_bulk_bytes=args.max_bulk_bytes,
        )
        for job in jobs
    ]
    summary = {
        "endpoint_file": str(args.endpoint_file),
        "index_prefix": args.index_prefix,
        "batch_size": args.batch_size,
        "max_bulk_bytes": args.max_bulk_bytes,
        "resume_existing": args.resume_existing,
        "results": results,
    }
    args.summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/batch2_keyframes.py
"""Batch-2 keyframe registry contract (M news, N traffic cameras, S01 cycling).

Shared by the two batch-2 uploaders:

* ``elastic_upload_batch2_keyframe_map.py`` — keyframe map → Elastic
* ``milvus_upload_pe_core_batch2.py``       — PE-Core-G14-448 vectors → Milvus

The source of truth is the three audited ``final/frame_registry.parquet`` files
(``Batch2/AIC2026_Batch2_M_N_Keyframe_R2_HANDOFF.md`` §5.1, §12.1), one row per
JPEG on R2. Each is pinned by SHA-256, so an uploader can never mix rows from a
re-run keyframe snapshot with vectors encoded from the old one.

Identity differs from L in two ways, and both are load-bearing:

* the JPEG on R2 is named by the ordinal ``n`` (``001.jpg``), not by
  ``frame_idx``; the application id is still ``<category>/<video_id>/<n:03d>``;
* N and S01 video ids use a hyphen (``N001-V001``, ``S01-V001``), M an
  underscore (``M01_V001``). The category is therefore never ``split("_")``
  of the video id — it is matched per profile below.
"""

from __future__ import annotations

import hashlib
import math
import re
from dataclasses import dataclass
from pathlib import Path

DEFAULT_REGISTRY_ROOT = Path("/home/bao/Projects/ExtractKeyframe/keyframe_batch2")
R2_BUCKET = "aic26-infoshot-keyframes"
#: Frames below this were not embedded: black tail of M10_V029 and fades (handoff §11.1–11.2).
MIN_QUALITY = 0.05
REGISTRY_COLUMNS = (
    "n",
    "frame_id",
    "video_id",
    "category",
    "frame_idx",
    "pts_time",
    "fps",
    "quality",
    "keyframe_name",
    "submit_keyframe_id",
    "r2_bucket",
    "r2_key",
    "pipeline_signature",
)


@dataclass(frozen=True)
class Profile:
    name: str
    directory: str
    pipeline_signature: str
    registry_sha256: str
    rows: int
    videos: int
    categories: int
    #: Full match of a video id; group 1 is its category.
    video_pattern: re.Pattern[str]

    @property
    def registry_relpath(self) -> str:
        return f"{self.directory}/final/frame_registry.parquet"


PROFILES: dict[str, Profile] = {
    profile.name: profile
    for profile in (
        Profile(
            name="M",
            directory="infoshoot_m",
            pipeline_signature="e6c789322adb",
            registry_sha256="7b34dc36a3e69ae514e5d473a7c77645e71790a6ab24e47a486a20d8d8f460ae",
            rows=209_111,
            videos=304,
            categories=10,
            video_pattern=re.compile(r"(M(?:0[1-9]|10))_V\d{3}"),
        ),
        Profile(
            name="N",
            directory="infoshoot_n",
            pipeline_signature="773e4b7465c2",
            registry_sha256="523b1a11cadc0567233d47f7a46a5012eb1c8dc57df95ab0e00f40ac4d6b89fa",
            rows=44_572,
            videos=298,
            categories=100,
            video_pattern=re.compile(r"(N\d{3})-V\d{3}"),
        ),
        Profile(
            name="S",
            directory="infoshoot_s",
            pipeline_signature="db5918a110cf",
            registry_sha256="b3d7c630a2e71f04a445fbe6ab5020568ce098fca70fc34f911f20bc5a024a54",
            rows=621_117,
            videos=12,
            categories=1,
            video_pattern=re.compile(r"(S01)-V\d{3}"),
        ),
    )
}
TOTAL_KEYFRAMES = sum(profile.rows for profile in PROFILES.values())  # 874,800


@dataclass(frozen=True, slots=True)
class KeyframeRow:
    profile: str
    frame_id: str
    video_id: str
    category: str
    n: int
    frame_idx: int
    pts_time: float
    fps: float
    quality: float
    r2_key: str

    @property
    def keyframe_name(self) -> str:
        return f"{self.n:03d}"

    @property
    def keyframe_id(self) -> str:
        return f"{self.video_id}/{self.keyframe_name}"

    @property
    def submit_keyframe_id(self) -> str:
        return f"{self.category}/{self.keyframe_id}"

    @property
    def embedded(self) -> bool:
        return self.quality >= MIN_QUALITY


def sha256_file(path: Path, *, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def profile_for_video(video_id: str) -> tuple[Profile, str]:
    """The profile and category of a batch-2 video id, or ValueError."""
    for profile in PROFILES.values():
        match = profile.video_pattern.fullmatch(video_id)
        if match:
            return profile, match.group(1)
    raise ValueError(f"Not a batch-2 video id: {video_id!r}")


def _finite(value: float) -> bool:
    return isinstance(value, float) and math.isfinite(value)


def validate_rows(profile: Profile, columns: dict[str, list]) -> list[KeyframeRow]:
    """Check every row against the identity contract; return rows ordered by (video, n)."""
    count = len(columns["frame_id"])
    rows: list[KeyframeRow] = []
    seen_frame_ids: set[str] = set()
    for index in range(count):
        value = {name: columns[name][index] for name in REGISTRY_COLUMNS}
        frame_id = str(value["frame_id"])
        video_id = str(value["video_id"])
        where = f"{profile.registry_relpath} row {index} ({frame_id})"
        matched = profile.video_pattern.fullmatch(video_id)
        if not matched:
            raise ValueError(f"{where}: video_id {video_id!r} does not belong to profile {profile.name}")
        category = matched.group(1)
        n = int(value["n"])
        frame_idx = int(value["frame_idx"])
        pts_time = float(value["pts_time"])
        fps = float(value["fps"])
        quality = float(value["quality"])
        expected = {
            "category": category,
            "frame_id": f"{video_id}@f{frame_idx:08d}",
            "keyframe_name": f"{n:03d}.jpg",
            "submit_keyframe_id": f"{category}/{video_id}/{n:03d}",
            "r2_bucket": R2_BUCKET,
            "r2_key": f"Keyframes/Keyframes_{category}/{video_id}/{n:03d}.jpg",
            "pipeline_signature": profile.pipeline_signature,
        }
        for name, wanted in expected.items():
            if value[name] != wanted:
                raise ValueError(f"{where}: {name}={value[name]!r}, expected {wanted!r}")
        if n < 1 or frame_idx < 0:
            raise ValueError(f"{where}: invalid n={n} / frame_idx={frame_idx}")
        if not (_finite(pts_time) and _finite(fps) and fps > 0 and _finite(quality)):
            raise ValueError(f"{where}: invalid pts_time/fps/quality")
        if frame_id in seen_frame_ids:
            raise ValueError(f"{where}: duplicate frame_id")
        seen_frame_ids.add(frame_id)
        rows.append(
            KeyframeRow(
                profile=profile.name,
                frame_id=frame_id,
                video_id=video_id,
                category=category,
                n=n,
                frame_idx=frame_idx,
                pts_time=pts_time,
                fps=fps,
                quality=quality,
                r2_key=value["r2_key"],
            )
        )

    rows.sort(key=lambda row: (row.video_id, row.n))
    previous: KeyframeRow | None = None
    for row in rows:
        if previous is None or previous.video_id != row.video_id:
            if row.n != 1:
                raise ValueError(f"{row.video_id}: first keyframe is n={row.n}, expected 1")
        else:
            if row.n != previous.n + 1:
                raise ValueError(f"{row.video_id}: n jumps {previous.n} -> {row.n}")
            if row.frame_idx <= previous.frame_idx:
                raise ValueError(f"{row.video_id}: frame_idx not strictly increasing at n={row.n}")
        previous = row
    return rows


def load_registry(profile: Profile, root: Path = DEFAULT_REGISTRY_ROOT) -> list[KeyframeRow]:
    """Read one pinned registry and prove it is exactly the audited snapshot."""
    try:
        import pyarrow.parquet as pq
    except ModuleNotFoundError as exc:
        raise SystemExit("Missing pyarrow. Install it with: uv pip install pyarrow") from exc

    path = root / profile.registry_relpath
    if not path.is_file():
        raise FileNotFoundError(path)
    digest = sha256_file(path)
    if digest != profile.registry_sha256:
        raise ValueError(
            f"{path}: SHA-256 {digest[:12]}… differs from the audited snapshot "
            f"{profile.registry_sha256[:12]}… — the keyframe snapshot changed; do not upload."
        )
    table = pq.read_table(path, columns=list(REGISTRY_COLUMNS))
    rows = validate_rows(profile, {name: table.column(name).to_pylist() for name in REGISTRY_COLUMNS})
    videos = {row.video_id for row in rows}
    categories = {row.category for row in rows}
    if len(rows) != profile.rows or len(videos) != profile.videos or len(categories) != profile.categories:
        raise ValueError(
            f"{path}: {len(rows):,} rows / {len(videos)} videos / {len(categories)} categories, expected "
            f"{profile.rows:,} / {profile.videos} / {profile.categories}"
        )
    return rows


def load_registries(
    names: tuple[str, ...] = tuple(PROFILES), root: Path = DEFAULT_REGISTRY_ROOT
) -> dict[str, list[KeyframeRow]]:
    result: dict[str, list[KeyframeRow]] = {}
    for name in names:
        rows = load_registry(PROFILES[name], root)
        embedded = sum(row.embedded for row in rows)
        print(
            f"[registry {name}] {len(rows):,} keyframes, {len({row.video_id for row in rows})} videos, "
            f"{embedded:,} at quality ≥ {MIN_QUALITY} — SHA-256 pinned, identity contract PASS",
            flush=True,
        )
        result[name] = rows
    return result

In [ ]:
%%writefile /content/r2_publish_n_web_videos.py
#!/usr/bin/env python3
"""Publish browser-playable copies of the batch-2 traffic-camera (N) videos on R2.

The N recordings were remuxed from the cameras' .mov files and still carry the
cameras' SEI messages, whose ``payload_size`` overruns their NAL unit. FFmpeg's
lenient default decoder skips them, but Chrome rejects the stream
(``PIPELINE_ERROR_DECODE`` on the third packet) and the console shows a black
player. Stripping SEI with a bitstream filter fails for the same reason: the
filter must parse the SEI to drop it.

This script instead overwrites every SEI NAL unit with a filler-data NAL unit
(type 12, ignored by every decoder) of exactly the same length. Nothing the
container describes moves — sample sizes, offsets, timestamps and every decoded
frame stay identical, so ``frame_idx``/``pts_time`` in the keyframe maps remain
valid — and the stream decodes under strict error detection.

Originals are never touched. Each copy goes to a new key that mirrors the layout:

    aic26-media/Videos/Videos_N001/N001-V001.mp4      (BTC original, unchanged)
    aic26-media/Videos_Web/Videos_N001/N001-V001.mp4  (this script's output)

A new key rather than an in-place overwrite, because the video host serves
``Cache-Control: public, max-age=31536000, immutable``: an overwritten object
would stay hidden behind the cached broken bytes, at the CDN and in browsers.

Per video: download, check size, record the packet table, patch, require the
packet table to be byte-identical and strict decoding to lose no more frames
than the lenient decoder does with no SEI complaint left (an unpatched recording
decodes about a third of them; what both lose is the cameras' own frame damage,
which Chrome conceals), upload with
provenance metadata, then HEAD the copy. A copy whose ``source-etag`` matches
the current original is skipped, so rerunning the command resumes.

Run: ``uv run --with boto3 python r2_publish_n_web_videos.py --work-dir <tmp>``
"""

from __future__ import annotations

import argparse
import csv
import json
import re
import subprocess
import sys
import tempfile
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

from elastic_upload import read_r2_credentials

BUCKET = "aic26-media"
SOURCE_PREFIX = "Videos/"
TARGET_PREFIX = "Videos_Web/"
SOURCE_KEY = re.compile(r"Videos/Videos_(N\d{3})/(N\d{3})-V\d{3}\.mp4")
PIPELINE = "aic2026-n-sei-filler-v1"
NAL_LENGTH_SIZE = 4  # avcC lengthSizeMinusOne = 3 in every N recording
SEI, FILLER = 6, 12


def target_key(source_key: str) -> str:
    return TARGET_PREFIX + source_key[len(SOURCE_PREFIX):]


def packet_table(path: Path) -> list[list[str]]:
    """Every video packet's pts, dts, duration, size, flags and file position."""
    out = subprocess.run(
        [
            "ffprobe", "-v", "quiet", "-select_streams", "v:0",
            "-show_entries", "packet=pts,dts,duration,size,pos,flags", "-of", "csv=p=0", str(path),
        ],
        capture_output=True, text=True, check=True,
    ).stdout
    return list(csv.reader(out.splitlines()))


def replace_sei_with_filler(path: Path, packets: list[list[str]]) -> Counter[str]:
    """Rewrite SEI NAL units as same-length filler, in place; returns NAL type counts."""
    stats: Counter[str] = Counter()
    # ffprobe prints the requested fields in its own order: pts, dts, duration, size, pos, flags.
    with path.open("r+b") as handle:
        for _, _, _, size_text, pos_text, _ in packets:
            size, pos = int(size_text), int(pos_text)
            handle.seek(pos)
            data = bytearray(handle.read(size))
            offset, changed = 0, False
            while offset < size:
                length = int.from_bytes(data[offset:offset + NAL_LENGTH_SIZE], "big")
                offset += NAL_LENGTH_SIZE
                if length <= 1 or offset + length > size:
                    raise ValueError(f"{path.name}: bad NAL length {length} at file offset {pos + offset}")
                nal_type = data[offset] & 0x1F
                stats[f"nal_{nal_type}"] += 1
                if nal_type == SEI:
                    # forbidden_zero_bit 0, nal_ref_idc 0, type 12; ff_bytes; rbsp_trailing_bits.
                    data[offset] = FILLER
                    data[offset + 1:offset + length - 1] = b"\xff" * (length - 2)
                    data[offset + length - 1] = 0x80
                    changed = True
                offset += length
            if changed:
                handle.seek(pos)
                handle.write(data)
                stats["sei_replaced"] += 1
    return stats


def count_frames(path: Path, *, strict: bool) -> tuple[int, subprocess.CompletedProcess[str]]:
    """Decode the whole stream and count frames; ffprobe does it without a muxer.

    (An ffmpeg `-f null` run also reports the cameras' repeated DTS values, a
    muxer complaint about the unchanged timeline, not a decoding error.)
    """
    result = subprocess.run(
        ["ffprobe", "-v", "error", *(["-err_detect", "explode"] if strict else []),
         "-select_streams", "v:0", "-count_frames", "-show_entries", "stream=nb_read_frames",
         "-of", "csv=p=0", str(path)],
        capture_output=True, text=True,
    )
    return int(result.stdout.strip() or 0), result


def strict_decode(path: Path, expected_frames: int) -> int:
    """Strict decoding must lose nothing the recording had not already lost.

    Some cameras dropped reference frames while recording (`mmco: unref short
    failure`); even FFmpeg's lenient decoder loses those frames from the original
    (N007-V003: 6,319 of 6,610), and Chrome conceals them and keeps playing. So
    the patched copy is compared with its own lenient decode, not with the packet
    count. A recording whose SEI is still broken decodes ~30 % under strict rules.
    """
    strict, result = count_frames(path, strict=True)
    lenient, _ = count_frames(path, strict=False)
    # Strict decoding may drop a few more of the damaged frames than the lenient
    # decoder conceals (N027-V001: 19 of 14,723); 1 % stays far from the ~70 %
    # a recording loses while its SEI is still broken.
    tolerance = max(2, expected_frames // 100)
    if result.returncode != 0 or "SEI" in result.stderr or strict < lenient - tolerance:
        raise ValueError(
            f"{path.name}: strict decode gave {strict} frames, lenient {lenient}, packets "
            f"{expected_frames}: {result.stderr.strip()[:300]}"
        )
    return strict


def process(s3: Any, source_key: str, work_dir: Path, *, dry_run: bool) -> dict[str, Any]:
    video_id = source_key.rsplit("/", 1)[1][:-4]
    source = s3.head_object(Bucket=BUCKET, Key=source_key)
    destination = target_key(source_key)
    try:
        existing = s3.head_object(Bucket=BUCKET, Key=destination)
    except s3.exceptions.ClientError:
        existing = None
    if existing and existing.get("Metadata", {}).get("source-etag") == source["ETag"] and \
            existing["Metadata"].get("pipeline") == PIPELINE:
        return {"video_id": video_id, "status": "skipped"}
    if dry_run:
        return {"video_id": video_id, "status": "planned", "size": source["ContentLength"]}

    started = time.monotonic()
    local = work_dir / f"{video_id}.mp4"
    try:
        s3.download_file(BUCKET, source_key, str(local))
        if local.stat().st_size != source["ContentLength"]:
            raise ValueError(f"{video_id}: downloaded {local.stat().st_size} bytes, expected {source['ContentLength']}")
        before = packet_table(local)
        stats = replace_sei_with_filler(local, before)
        if packet_table(local) != before:
            raise ValueError(f"{video_id}: packet table changed after patching")
        strict_frames = strict_decode(local, len(before))
        s3.upload_file(
            str(local), BUCKET, destination,
            ExtraArgs={
                "ContentType": source.get("ContentType", "video/mp4"),
                "CacheControl": source.get("CacheControl", "public, max-age=31536000, immutable"),
                "Metadata": {
                    "pipeline": PIPELINE,
                    "source-key": source_key,
                    "source-etag": source["ETag"],
                    "source-size": str(source["ContentLength"]),
                    "video-frames": str(len(before)),
                    "sei-units-replaced": str(stats["sei_replaced"]),
                    "strict-decoded-frames": str(strict_frames),
                },
            },
        )
        uploaded = s3.head_object(Bucket=BUCKET, Key=destination)
        if uploaded["ContentLength"] != source["ContentLength"]:
            raise ValueError(f"{video_id}: uploaded size {uploaded['ContentLength']} differs from source")
        return {
            "video_id": video_id,
            "status": "published",
            "key": destination,
            "frames": len(before),
            "strict_decoded_frames": strict_frames,
            "sei_replaced": stats["sei_replaced"],
            "seconds": round(time.monotonic() - started, 1),
        }
    finally:
        local.unlink(missing_ok=True)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--credentials-file", type=Path, default=Path("CloudflareR2/cloudflareR2_api.txt"))
    parser.add_argument("--work-dir", type=Path, default=Path(tempfile.gettempdir()) / "aic26_n_web_videos")
    parser.add_argument("--workers", type=int, default=3)
    parser.add_argument("--only", nargs="*", help="Process only these video ids (e.g. N070-V002).")
    parser.add_argument("--report", type=Path, default=Path("r2_publish_n_web_videos_report.json"))
    parser.add_argument("--dry-run", action="store_true", help="List what would be published; no download or write.")
    args = parser.parse_args()

    import boto3  # noqa: PLC0415 — only this script needs it (uv run --with boto3)

    creds = read_r2_credentials(args.credentials_file)
    s3 = boto3.client(
        "s3", endpoint_url=creds["endpoint"], aws_access_key_id=creds["access_key"],
        aws_secret_access_key=creds["secret_key"], region_name="auto",
    )
    keys = [
        item["Key"]
        for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix="Videos/Videos_N")
        for item in page.get("Contents", [])
        if SOURCE_KEY.fullmatch(item["Key"])
    ]
    if args.only:
        wanted = set(args.only)
        keys = [key for key in keys if key.rsplit("/", 1)[1][:-4] in wanted]
    print(f"{len(keys)} N videos -> s3://{BUCKET}/{TARGET_PREFIX}", flush=True)
    args.work_dir.mkdir(parents=True, exist_ok=True)

    results: list[dict[str, Any]] = []
    with ThreadPoolExecutor(args.workers) as pool:
        futures = {pool.submit(process, s3, key, args.work_dir, dry_run=args.dry_run): key for key in keys}
        for future in as_completed(futures):
            try:
                result = future.result()
            except Exception as exc:  # noqa: BLE001 — one bad video must not stop the rest
                video_id = futures[future].rsplit("/", 1)[1][:-4]
                result = {"video_id": video_id, "status": "failed", "error": f"{type(exc).__name__}: {exc}"}
            results.append(result)
            print(f"[{len(results)}/{len(keys)}] {json.dumps(result)}", flush=True)

    summary = Counter(result["status"] for result in results)
    args.report.write_text(
        json.dumps({"finished_at": datetime.now(UTC).isoformat(), "pipeline": PIPELINE, "summary": summary,
                    "results": sorted(results, key=lambda r: r["video_id"])}, indent=2) + "\n",
        encoding="utf-8",
    )
    print(f"DONE {dict(summary)} (report: {args.report})", flush=True)
    return 1 if summary.get("failed") else 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /content/r2_reencode_n_damaged_videos.py
#!/usr/bin/env python3
"""Re-encode the N traffic-camera videos whose own recordings are damaged.

`r2_publish_n_web_videos.py` fixed the malformed SEI of every N video, but 103
recordings also lost reference frames while the cameras were writing them
(`mmco: unref short failure`). Chrome conceals most of that damage, yet on some
keyframes it stops the whole pipeline (N027-V003 at 4:34), so those videos need
a clean bitstream, not a patched one.

Per video: decode the BTC original leniently (FFmpeg conceals the damage, as
the keyframe extraction's PyAV decode did), encode H.264 with NVENC keeping
every frame's original timestamp (`-copyts`, `-fps_mode passthrough`, the
source time base), then require:

* a strict decode of the output to be clean and to cover every packet;
* the output's frame times to equal the lenient decode of the source (≤ 2 ms);
* every keyframe of that video in the pinned batch-2 registry to have an output
  frame at its `pts_time` (≤ 2 ms), so seeking to a keyframe lands on it.

Copies go to a new key, `Videos_Web_v2/Videos_Nxxx/<id>.mp4`: the SEI-patched
copies under `Videos_Web/` are served `immutable` for a year, so overwriting
them would stay hidden behind the CDN. After each upload the video id is added
to the backend's `video_overrides.json`, which the media URL builder re-reads
on change — each video switches over as soon as it is published.

Run: ``uv run --with boto3 python r2_reencode_n_damaged_videos.py --videos-file <json> --work-dir <tmp>``
"""

from __future__ import annotations

import argparse
import bisect
import json
import os
import subprocess
import sys
import tempfile
import threading
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

from batch2_keyframes import DEFAULT_REGISTRY_ROOT, PROFILES, load_registry
from elastic_upload import read_r2_credentials
from r2_publish_n_web_videos import BUCKET, count_frames

TARGET_ROOT = "Videos_Web_v2"
PIPELINE = "aic2026-n-nvenc-reencode-v1"
DEFAULT_OVERRIDES = Path("backend/app/video_overrides.json")
TIME_TOLERANCE = 0.002  # seconds
_overrides_lock = threading.Lock()


def source_key(video_id: str) -> str:
    return f"Videos/Videos_{video_id.split('-')[0]}/{video_id}.mp4"


def target_key(video_id: str) -> str:
    return f"{TARGET_ROOT}/Videos_{video_id.split('-')[0]}/{video_id}.mp4"


def probe(path: Path, *args: str) -> str:
    return subprocess.run(
        ["ffprobe", "-v", "quiet", "-select_streams", "v:0", *args, "-of", "csv=p=0", str(path)],
        capture_output=True, text=True, check=True,
    ).stdout


def frame_times(path: Path) -> list[float]:
    """Times of the frames a lenient decode actually produces, in order."""
    out = probe(path, "-show_entries", "frame=best_effort_timestamp_time")
    return [float(v) for v in (line.strip().rstrip(",") for line in out.splitlines()) if v not in ("", "N/A")]


def packet_times(path: Path) -> list[float]:
    out = probe(path, "-show_entries", "packet=pts_time")
    return sorted(float(v) for v in (line.strip().rstrip(",") for line in out.splitlines()) if v not in ("", "N/A"))


def reencode(src: Path, out: Path) -> None:
    time_base = probe(src, "-show_entries", "stream=time_base").strip()
    timescale = time_base.split("/")[1] if "/" in time_base else "90000"
    result = subprocess.run(
        [
            "ffmpeg", "-hide_banner", "-nostdin", "-v", "error", "-y", "-copyts", "-i", str(src),
            "-map", "0:v:0", "-an", "-sn", "-dn",
            "-c:v", "h264_nvenc", "-preset", "p5", "-tune", "hq", "-profile:v", "high",
            "-rc", "vbr", "-cq", "25", "-b:v", "0", "-maxrate", "6M", "-bufsize", "12M",
            "-g", "25", "-bf", "0", "-pix_fmt", "yuv420p",
            "-fps_mode", "passthrough", "-enc_time_base", "demux", "-video_track_timescale", timescale,
            "-movflags", "+faststart", str(out),
        ],
        capture_output=True, text=True,
    )
    if result.returncode != 0 or not out.exists():
        raise RuntimeError(f"{src.name}: ffmpeg failed: {result.stderr.strip()[-300:]}")


def max_gap(expected: list[float], actual: list[float]) -> float:
    """Largest distance from an expected time to the nearest actual one."""
    worst = 0.0
    for value in expected:
        at = bisect.bisect_left(actual, value)
        nearest = min(
            (abs(actual[i] - value) for i in (at - 1, at) if 0 <= i < len(actual)), default=float("inf")
        )
        worst = max(worst, nearest)
    return worst


def write_overrides(path: Path, video_ids: set[str]) -> None:
    data = {
        "about": (
            "Videos republished under their own R2 prefix (see r2_reencode_n_damaged_videos.py); "
            "MediaUrlBuilder.video_url serves these ids from that root."
        ),
        "roots": {TARGET_ROOT: sorted(video_ids)},
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{os.getpid()}.tmp")
    temporary.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    temporary.replace(path)


def add_override(path: Path, video_id: str) -> None:
    with _overrides_lock:
        data = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
        write_overrides(path, set((data.get("roots") or {}).get(TARGET_ROOT, [])) | {video_id})


def published_copies(s3: Any) -> set[str]:
    """Video ids whose `Videos_Web_v2/` copy was made from the current original.

    This is the source of truth when several machines re-encode in parallel: a
    copy another machine uploaded counts as soon as it is on R2.
    """
    keys = [
        item["Key"]
        for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=f"{TARGET_ROOT}/")
        for item in page.get("Contents", [])
        if item["Key"].endswith(".mp4")
    ]

    def valid(key: str) -> str | None:
        video_id = key.rsplit("/", 1)[1][:-4]
        meta = s3.head_object(Bucket=BUCKET, Key=key).get("Metadata", {})
        source = s3.head_object(Bucket=BUCKET, Key=source_key(video_id))
        return video_id if meta.get("pipeline") == PIPELINE and meta.get("source-etag") == source["ETag"] else None

    with ThreadPoolExecutor(16) as pool:
        return {video for video in pool.map(valid, keys) if video}


def process(s3: Any, video_id: str, keyframes: dict[str, list[float]], work_dir: Path, overrides: Path) -> dict[str, Any]:
    source = s3.head_object(Bucket=BUCKET, Key=source_key(video_id))
    try:
        existing = s3.head_object(Bucket=BUCKET, Key=target_key(video_id))
    except s3.exceptions.ClientError:
        existing = None
    if existing and existing["Metadata"].get("source-etag") == source["ETag"] and \
            existing["Metadata"].get("pipeline") == PIPELINE:
        add_override(overrides, video_id)
        return {"video_id": video_id, "status": "skipped"}

    started = time.monotonic()
    src, out = work_dir / f"{video_id}.src.mp4", work_dir / f"{video_id}.mp4"
    try:
        s3.download_file(BUCKET, source_key(video_id), str(src))
        if src.stat().st_size != source["ContentLength"]:
            raise ValueError(f"{video_id}: downloaded size differs from the source")
        expected = frame_times(src)
        reencode(src, out)
        actual = packet_times(out)
        if len(actual) != len(expected):
            raise ValueError(f"{video_id}: output has {len(actual)} frames, lenient source decode {len(expected)}")
        timeline_gap = max_gap(expected, actual)
        keyframe_gap = max_gap(keyframes.get(video_id, []), actual)
        if timeline_gap > TIME_TOLERANCE or keyframe_gap > TIME_TOLERANCE:
            raise ValueError(f"{video_id}: timestamps moved (timeline {timeline_gap:.4f}s, keyframes {keyframe_gap:.4f}s)")
        strict, result = count_frames(out, strict=True)
        if result.returncode != 0 or result.stderr.strip() or strict != len(actual):
            raise ValueError(f"{video_id}: output strict decode {strict}/{len(actual)}: {result.stderr.strip()[:200]}")
        s3.upload_file(
            str(out), BUCKET, target_key(video_id),
            ExtraArgs={
                "ContentType": "video/mp4",
                "CacheControl": "public, max-age=31536000, immutable",
                "Metadata": {
                    "pipeline": PIPELINE,
                    "source-key": source_key(video_id),
                    "source-etag": source["ETag"],
                    "source-size": str(source["ContentLength"]),
                    "video-frames": str(len(actual)),
                    "keyframes-checked": str(len(keyframes.get(video_id, []))),
                },
            },
        )
        uploaded = s3.head_object(Bucket=BUCKET, Key=target_key(video_id))
        if uploaded["ContentLength"] != out.stat().st_size:
            raise ValueError(f"{video_id}: uploaded size differs from the encoded file")
        add_override(overrides, video_id)
        return {
            "video_id": video_id, "status": "published", "key": target_key(video_id),
            "frames": len(actual), "keyframes_checked": len(keyframes.get(video_id, [])),
            "max_keyframe_gap_ms": round(keyframe_gap * 1000, 3),
            "mb": round(out.stat().st_size / 1e6, 1), "seconds": round(time.monotonic() - started, 1),
        }
    finally:
        src.unlink(missing_ok=True)
        out.unlink(missing_ok=True)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--videos-file", type=Path, help='JSON with {"damaged": [video ids]}')
    parser.add_argument("--only", nargs="*")
    parser.add_argument("--registry-root", type=Path, default=DEFAULT_REGISTRY_ROOT)
    parser.add_argument("--reverse", action="store_true", help="Work from the end of the list (second machine).")
    parser.add_argument(
        "--sync-overrides", action="store_true",
        help="Only rewrite the overrides file from the copies on R2 (no encoding).",
    )
    parser.add_argument("--credentials-file", type=Path, default=Path("CloudflareR2/cloudflareR2_api.txt"))
    parser.add_argument("--overrides-file", type=Path, default=DEFAULT_OVERRIDES)
    parser.add_argument("--work-dir", type=Path, default=Path(tempfile.gettempdir()) / "aic26_n_reencode")
    parser.add_argument("--workers", type=int, default=3)
    parser.add_argument("--report", type=Path, default=Path("r2_reencode_n_damaged_videos_report.json"))
    args = parser.parse_args()

    import boto3  # noqa: PLC0415 — uv run --with boto3

    creds = read_r2_credentials(args.credentials_file)
    s3 = boto3.client("s3", endpoint_url=creds["endpoint"], aws_access_key_id=creds["access_key"],
                      aws_secret_access_key=creds["secret_key"], region_name="auto")
    if args.sync_overrides:
        published = published_copies(s3)
        with _overrides_lock:
            write_overrides(args.overrides_file, published)
        print(f"{args.overrides_file}: {len(published)} videos served from {TARGET_ROOT}/", flush=True)
        return 0
    if args.videos_file is None:
        parser.error("--videos-file is required unless --sync-overrides")
    videos = json.loads(args.videos_file.read_text(encoding="utf-8"))["damaged"]
    if args.only:
        videos = [video for video in videos if video in set(args.only)]
    if args.reverse:
        videos = videos[::-1]
    keyframes: dict[str, list[float]] = {}
    for row in load_registry(PROFILES["N"], args.registry_root):
        keyframes.setdefault(row.video_id, []).append(row.pts_time)
    args.work_dir.mkdir(parents=True, exist_ok=True)
    print(f"{len(videos)} damaged N videos -> s3://{BUCKET}/{TARGET_ROOT}/", flush=True)

    results: list[dict[str, Any]] = []
    with ThreadPoolExecutor(args.workers) as pool:
        futures = {pool.submit(process, s3, video, keyframes, args.work_dir, args.overrides_file): video for video in videos}
        for future in as_completed(futures):
            try:
                result = future.result()
            except Exception as exc:  # noqa: BLE001 — one bad video must not stop the rest
                result = {"video_id": futures[future], "status": "failed", "error": f"{type(exc).__name__}: {exc}"}
            results.append(result)
            print(f"[{len(results)}/{len(videos)}] {json.dumps(result)}", flush=True)

    summary = Counter(result["status"] for result in results)
    args.report.write_text(json.dumps({"finished_at": datetime.now(UTC).isoformat(), "pipeline": PIPELINE,
                                       "summary": summary, "results": sorted(results, key=lambda r: r["video_id"])},
                                      indent=2) + "\n", encoding="utf-8")
    print(f"DONE {dict(summary)} (report: {args.report})", flush=True)
    return 1 if summary.get("failed") else 0


if __name__ == "__main__":
    sys.exit(main())

## 4. Registry keyframe N (SHA-256 được script kiểm) và danh sách video

In [ ]:
import json, urllib.request
from pathlib import Path
registry = Path("/content/keyframe_batch2/infoshoot_n/final/frame_registry.parquet")
registry.parent.mkdir(parents=True, exist_ok=True)
if not registry.exists():
    # Cloudflare answers the default Python-urllib User-Agent with 403.
    request = urllib.request.Request("https://pub-5010c807b73743ba82d6c40852ae9e6b.r2.dev/manifest/infoshoot_n/final/frame_registry.parquet", headers={"User-Agent": "Mozilla/5.0 (aic26-colab)"})
    with urllib.request.urlopen(request, timeout=120) as response:
        registry.write_bytes(response.read())
Path("/content/n_damaged.json").write_text(json.dumps({"damaged": ["N004-V001", "N004-V002", "N004-V003", "N007-V001", "N007-V002", "N007-V003", "N008-V001", "N008-V002", "N008-V003", "N009-V001", "N009-V002", "N015-V001", "N015-V002", "N015-V003", "N019-V003", "N021-V001", "N021-V002", "N021-V003", "N027-V001", "N027-V002", "N027-V003", "N028-V001", "N028-V002", "N028-V003", "N029-V001", "N029-V002", "N029-V003", "N030-V001", "N030-V002", "N030-V003", "N032-V001", "N032-V002", "N032-V003", "N039-V001", "N039-V002", "N039-V003", "N040-V001", "N040-V002", "N040-V003", "N041-V002", "N041-V003", "N043-V004", "N043-V005", "N045-V001", "N045-V002", "N045-V003", "N055-V001", "N055-V003", "N058-V001", "N058-V002", "N058-V003", "N059-V001", "N059-V002", "N059-V003", "N060-V001", "N063-V002", "N066-V001", "N066-V002", "N066-V003", "N067-V001", "N067-V002", "N067-V003", "N069-V001", "N069-V002", "N069-V003", "N073-V001", "N073-V002", "N073-V003", "N076-V001", "N076-V002", "N076-V003", "N077-V002", "N077-V003", "N079-V002", "N083-V001", "N083-V002", "N083-V003", "N085-V001", "N085-V002", "N087-V001", "N087-V002", "N087-V003", "N088-V001", "N088-V002", "N088-V003", "N089-V001", "N089-V002", "N089-V003", "N092-V001", "N092-V002", "N092-V003", "N093-V001", "N093-V002", "N093-V003", "N095-V001", "N095-V002", "N095-V003", "N096-V001", "N096-V002", "N096-V003", "N097-V001", "N097-V002", "N097-V003"]}))
print(registry.stat().st_size, "bytes registry;", 103, "videos")

## 5. Chạy (từ cuối danh sách). Chạy lại ô này để tiếp tục nếu bị ngắt.

In [ ]:
%cd /content
!python -u r2_reencode_n_damaged_videos.py --videos-file /content/n_damaged.json --reverse --workers 4 \
    --registry-root /content/keyframe_batch2 --work-dir /content/work \
    --overrides-file /content/video_overrides.json --report /content/reencode_report.json

In [ ]:
import json
report = json.load(open("/content/reencode_report.json"))
print(report["summary"])
for row in report["results"]:
    if row["status"] == "failed":
        print(row["video_id"], row["error"][:200])